# Solution 02 on Google Colab — VLM LoRA fine-tune, real data only

Runs the existing `solutions/02_vlm_lora_real_data/run_pipeline.py` unchanged (no code is duplicated here) against **Qwen/Qwen3-VL-8B-Instruct** (the script's own default). This notebook only handles the Colab-specific parts: mounting your Drive, copying the project onto local disk for fast I/O, installing dependencies, running the pipeline, and copying results back to Drive.

**Before running anything:** `Runtime > Change runtime type > T4/L4/A100 GPU`. Prefer an A100 if your Colab tier gives you one (see the compute-unit discussion elsewhere in this project) -- an 8B model in 4-bit fits comfortably on any of them, but training is much faster on an A100.

**Assumption about your Drive layout:** this notebook expects `solutions/`, `participant_kit/`, `data/`, and `shroom-vis-images/` to already exist together under one folder on your Drive (e.g. everything you'd otherwise `scp` to a VM, per `solutions/NLP2_RUNBOOK.md`). Set `DRIVE_PROJECT_PATH` below to point at that folder.

**Known limitation, be aware of it:** `train_lora.py` is a plain custom training loop (not HF `Trainer`), so it does **not** support mid-training checkpoint resume -- it only saves the adapter once, at the very end. If Colab disconnects mid-training, that run's progress is lost and you restart from scratch. Run `--smoke_test` first (a couple of minutes) to catch problems before committing to the multi-hour full run, and prefer Colab Pro+'s background execution to reduce disconnect risk on the long run.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# --- Config: edit these ---
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/SHROOM"   # <-- change to wherever your project actually lives on Drive
LOCAL_PROJECT_PATH = "/content/SHROOM"                  # local disk copy, much faster I/O than reading Drive directly
GPU_INDEX = 0        # Colab only ever exposes one GPU -- leave this at 0
EPOCHS = 2
GRAD_ACCUM = 8

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # cheap insurance against allocator fragmentation
print("DRIVE_PROJECT_PATH:", DRIVE_PROJECT_PATH)
print("LOCAL_PROJECT_PATH:", LOCAL_PROJECT_PATH)

DRIVE_PROJECT_PATH: /content/drive/MyDrive/SHROOM
LOCAL_PROJECT_PATH: /content/SHROOM


In [3]:
# Copy just the subfolders the pipeline actually needs from Drive to local disk.
# Skips anything already copied, so re-running this cell after a runtime restart is cheap.
os.makedirs(LOCAL_PROJECT_PATH, exist_ok=True)
for sub in ["data", "shroom-vis-images", "participant_kit", "solutions"]:
    src = f"{DRIVE_PROJECT_PATH}/{sub}"
    dst = f"{LOCAL_PROJECT_PATH}/{sub}"
    if os.path.exists(dst):
        print(f"skip (already present locally): {sub}")
        continue
    print(f"copying {sub} from Drive ... (this can take a few minutes for shroom-vis-images/)")
    !cp -r "{src}" "{dst}"
print("done")
!ls "{LOCAL_PROJECT_PATH}"

copying data from Drive ... (this can take a few minutes for shroom-vis-images/)
copying shroom-vis-images from Drive ... (this can take a few minutes for shroom-vis-images/)
copying participant_kit from Drive ... (this can take a few minutes for shroom-vis-images/)
copying solutions from Drive ... (this can take a few minutes for shroom-vis-images/)
done
data  participant_kit  shroom-vis-images  solutions


In [4]:
# Dependencies -- same list as solutions/NLP2_RUNBOOK.md. Deliberately NOT reinstalling
# torch/torchvision/torchaudio: Colab's preinstalled build already matches its own
# CUDA driver, and reinstalling a mismatched wheel is exactly what caused the CUDA
# version headaches on the nlp3 VM earlier in this project. We only check the version.
!pip install -q "git+https://github.com/huggingface/transformers"
!pip install -q "huggingface-hub<1.0" "accelerate>=0.34" "peft>=0.13" sentencepiece
!pip install -q bitsandbytes sacremoses

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 124.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 70.0 MB/s eta 0:00:00


In [5]:
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

major, minor = (int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
if (major, minor) < (2, 6):
    print("\ntorch is older than 2.6 -- transformers' torch.load safety check (CVE-2025-32434) will "
          "block model loading. Upgrading now:")
    !pip install -q -U torch torchvision torchaudio
    print("IMPORTANT: Colab often needs Runtime > Restart runtime for the upgraded torch to actually "
          "take effect (the old build may already be loaded in this process). Restart, then re-run "
          "this notebook from the top -- already-copied folders will be skipped automatically.")
else:
    print("torch version OK.")

torch: 2.11.0+cu128 | CUDA available: True
GPU: NVIDIA A100-SXM4-40GB | VRAM: 42.4 GB
torch version OK.


In [6]:
%cd {LOCAL_PROJECT_PATH}
# Sanity-check the local scorer wrapper matches participant_kit/scorer.py exactly
# before trusting anything this notebook produces later.
!python -m solutions.common.scorer_local

/content/SHROOM
Sanity check: gold predicted back against itself (expect ~1.0 everywhere)
n=3020  IoU=1.0000  Cor=0.9903  CorLbl=1.0000
  [en] n= 729  IoU=1.0000  Cor=0.9848  CorLbl=1.0000
  [fr] n= 754  IoU=1.0000  Cor=0.9914  CorLbl=1.0000
  [it] n= 757  IoU=1.0000  Cor=0.9956  CorLbl=1.0000
  [zh] n= 780  IoU=1.0000  Cor=0.9892  CorLbl=1.0000
  <invention           > n=1225  IoU=1.0000  Cor=1.0000
  <mischaracterization > n=1358  IoU=1.0000  Cor=1.0000
  <OCR                 > n= 179  IoU=1.0000  Cor=1.0000
  <miscounting         > n= 359  IoU=1.0000  Cor=1.0000
  <other               > n= 255  IoU=1.0000  Cor=1.0000
OK: scorer_local matches official scorer.py on the identity case.


In [7]:
%cd {LOCAL_PROJECT_PATH}/solutions/02_vlm_lora_real_data
# Data pipeline only, no model download -- fast, catches conversation-building/collation bugs early.
!python train_lora.py --dry_run

/content/SHROOM/solutions/02_vlm_lora_real_data
Loaded 12082 training records (device=cuda)
Built 12082 usable conversations
2026-07-30 09:42:18.090986: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-30 09:42:18.160748: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
preprocessor_config.json: 100% 390/390 [00:00<00:00, 2.82MB/s]
tokenizer_config.json: 10.9kB [00:00, 39.6MB/s]
vocab.json: 2.78MB [00:00, 46.1MB/s]
merges.txt: 1.67MB [00:00, 128MB/s]
tokenizer.json: 7.03MB [00:00, 178MB/s]
video_preprocessor_config

In [ ]:
# Tiny real run (loads the actual model). Do this before committing to the full run below.
!python run_pipeline.py --smoke_test --gpu_index {GPU_INDEX}

=== Training (LoRA, real data only) ===
Loaded 20 training records (device=cuda:0)
Built 20 usable conversations
2026-07-17 13:31:50.400070: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-17 13:31:50.470893: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading Qwen/Qwen3-VL-8B-Instruct via Qwen3VLForConditionalGeneration (4bit=True, dtype=torch.bfloat16)...
config.json: 1.47kB [00:00, 9.11MB/s]
model.safetensors.index.json: 67.8kB [00:00, 185MB/s]
Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
model-00002-of-000

Once the smoke test above finishes cleanly, run the full pipeline. This is the multi-hour step -- see the disconnect-risk note at the top of this notebook.

If you get assigned a smaller GPU (T4/L4, 16GB) instead of an A100 and hit a CUDA OOM (same failure mode worked through on the nlp3 VM earlier in this project), fall back to:
```
!python run_pipeline.py --gpu_index {GPU_INDEX} --epochs {EPOCHS} --grad_accum {GRAD_ACCUM} --max_pixels 401408 --target_modules q_proj,k_proj,v_proj,o_proj
```

In [ ]:
!python run_pipeline.py --gpu_index {GPU_INDEX} --epochs {EPOCHS} --grad_accum {GRAD_ACCUM}

=== Training (LoRA, real data only) ===
Loaded 12082 training records (device=cuda:0)
Built 12082 usable conversations
2026-07-17 13:36:03.069657: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-17 13:36:03.139748: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading Qwen/Qwen3-VL-8B-Instruct via Qwen3VLForConditionalGeneration (4bit=True, dtype=torch.bfloat16)...
Loading checkpoint shards:   0% 0/4 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _c

In [ ]:
# Copy results back to Drive so they survive this runtime being recycled.
dest = f"{DRIVE_PROJECT_PATH}/solutions/02_vlm_lora_real_data"
os.makedirs(dest, exist_ok=True)
for item in ["val_predictions.jsonl", "val_scores.json", "checkpoint"]:
    src = f"{LOCAL_PROJECT_PATH}/solutions/02_vlm_lora_real_data/{item}"
    if os.path.exists(src):
        print(f"copying {item} -> Drive ...")
        !cp -r "{src}" "{dest}/"
    else:
        print(f"skip (not found): {item}")
print("done -- check", dest, "in your Drive")

copying val_predictions.jsonl -> Drive ...
copying val_scores.json -> Drive ...
copying checkpoint -> Drive ...
done -- check /content/drive/MyDrive/SHROOM/solutions/02_vlm_lora_real_data in your Drive


In [ ]:
import json
with open(f"{LOCAL_PROJECT_PATH}/solutions/02_vlm_lora_real_data/val_scores.json") as f:
    print(json.dumps(json.load(f), indent=2))

{
  "n": 3020,
  "overall": {
    "iou": 0.43802181613070557,
    "cor": 0.47115240948899667,
    "cor_lbl": 0.41717630292159963
  },
  "per_language": {
    "en": {
      "iou": 0.37027758454685405,
      "cor": 0.41404982356922854,
      "cor_lbl": 0.3619773405601155,
      "n": 729
    },
    "fr": {
      "iou": 0.428140026460454,
      "cor": 0.45562671444997427,
      "cor_lbl": 0.39468134175131026,
      "n": 754
    },
    "it": {
      "iou": 0.44166850810424313,
      "cor": 0.4768274624305418,
      "cor_lbl": 0.4128283926932039,
      "n": 757
    },
    "zh": {
      "iou": 0.5073498525563844,
      "cor": 0.53402182502513,
      "cor_lbl": 0.4947309341098247,
      "n": 780
    }
  },
  "per_label": {
    "invention": {
      "iou": 0.18119850482710706,
      "cor": 0.203714303226623,
      "n_records_with_label": 1307
    },
    "mischaracterization": {
      "iou": 0.09423206436676279,
      "cor": 0.1183516850639173,
      "n_records_with_label": 1441
    },
    "OCR":

## Test-set predictions (for SHROOM website submission)

Same idea as solutions 03-06: everything above only ever predicted on `data/val.jsonl`. This section
runs the trained checkpoint over the real, **unlabeled** `data/test.jsonl` (4,898 examples) instead
-- the file you actually submit. No scoring step here, since test.jsonl has no gold labels.

**Budget real time for this.** Your own val run above measured ~5.79s/example on this exact
hardware (3,020 examples in 4h51m). Extrapolated to the 4,898 test examples, that's roughly **7-8
hours**. `predict.py` does **not** have the periodic-save/resume support `train_lora.py` now has --
it only writes `test_predictions.jsonl` once, at the very end. If the session gets cut off partway,
you lose the whole run and restart from scratch. Prefer Pro+'s background execution and a long,
uninterrupted session for this step -- ask if you want incremental-save/resume added to `predict.py`
first, same as was done for training, if that's a real risk for you.

**Required before submitting**: the SHROOM format needs one file per language, but
`data/test.jsonl` mixes all 4 together -- the cells below split the output and run the official
format checker to confirm it's actually submission-ready before you upload anything.

In [ ]:
sol02_local = f"{LOCAL_PROJECT_PATH}/solutions/02_vlm_lora_real_data"
test_predictions_path = f"{sol02_local}/test_predictions.jsonl"
checkpoint_path = f"{sol02_local}/checkpoint"
test_file_path = f"{LOCAL_PROJECT_PATH}/data/test.jsonl"

!python "{sol02_local}/predict.py" --checkpoint "{checkpoint_path}" --val_file "{test_file_path}" --output "{test_predictions_path}" --gpu_index {GPU_INDEX}

Loaded 4898 usable val conversations (device=cuda:0)
2026-07-30 09:42:37.666683: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-30 09:42:37.736275: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading base model Qwen/Qwen3-VL-8B-Instruct via Qwen3VLForConditionalGeneration (4bit=True, dtype=torch.bfloat16)...
config.json: 1.47kB [00:00, 10.2MB/s]
model.safetensors.index.json: 67.8kB [00:00, 194MB/s]
Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
model-00003-of-00004.safetensors:   0% 0.00/5.00G [00:00<?, ?B/s]



In [ ]:
split_script = f"{LOCAL_PROJECT_PATH}/solutions/common/split_predictions_by_language.py"
!python "{split_script}" "{test_predictions_path}"

  en: 1201 predictions -> /content/SHROOM/solutions/02_vlm_lora_real_data/test_predictions.en.jsonl
  fr: 1233 predictions -> /content/SHROOM/solutions/02_vlm_lora_real_data/test_predictions.fr.jsonl
  it: 1254 predictions -> /content/SHROOM/solutions/02_vlm_lora_real_data/test_predictions.it.jsonl
  zh: 1210 predictions -> /content/SHROOM/solutions/02_vlm_lora_real_data/test_predictions.zh.jsonl


In [ ]:
format_checker_script = f"{LOCAL_PROJECT_PATH}/participant_kit/format_checker.py"
reference_file = f"{LOCAL_PROJECT_PATH}/data/test.jsonl"
split_files = " ".join(
    f'"{sol02_local}/test_predictions.{lang}.jsonl"' for lang in ["en", "fr", "it", "zh"]
)
!python "{format_checker_script}" --reference-file "{reference_file}" {split_files}

Checked 4 file(s), 4898 row(s), 5540 span(s).
Languages: en, fr, it, zh
OK: submission format looks valid.


In [ ]:
# Copy the split, submission-ready files to Drive -- these are what you upload to the SHROOM site.
dest = f"{DRIVE_PROJECT_PATH}/solutions/02_vlm_lora_real_data"
os.makedirs(dest, exist_ok=True)
for lang in ["en", "fr", "it", "zh"]:
    src = f"{sol02_local}/test_predictions.{lang}.jsonl"
    if os.path.exists(src):
        print(f"copying test_predictions.{lang}.jsonl -> Drive ...")
        !cp "{src}" "{dest}/"
    else:
        print(f"skip (not found): test_predictions.{lang}.jsonl")
print("done -- check", dest, "in your Drive")

copying test_predictions.en.jsonl -> Drive ...
copying test_predictions.fr.jsonl -> Drive ...
copying test_predictions.it.jsonl -> Drive ...
copying test_predictions.zh.jsonl -> Drive ...
done -- check /content/drive/MyDrive/SHROOM/solutions/02_vlm_lora_real_data in your Drive


In [ ]:
# Quick sanity peek: line count + first record, so you can eyeball the format before submitting.
import json

with open(f"{sol02_local}/test_predictions.en.jsonl") as f:
    lines = f.readlines()
print(f"{len(lines)} EN predictions written")
print(json.dumps(json.loads(lines[0]), indent=2, ensure_ascii=False))

1201 EN predictions written
{
  "id": "test-en-428",
  "labels": []
}
